<a href="https://colab.research.google.com/github/CaoRIV/machine-learning-practicals/blob/main/01_Bai_tap_Gapminder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hệ thống bài tập Hồi quy tuyến tính với dữ liệu Gapminder

**Phiên bản người học**  
Biến đích chính: `lifeExp`. Mục tiêu dự báo: tuổi thọ kỳ vọng của các quốc gia ở các thời điểm tương lai.

Quy tắc quan trọng: chỉ dùng `train` để ước lượng, dùng `validation` để lựa chọn mô hình và chỉ mở `test` sau khi đã khóa mọi quyết định.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

#DATA_CANDIDATES = [Path("../data"), Path("data"), Path("."), Path("/content/")]
DATA_CANDIDATES = [
    Path("/content/drive/MyDrive/ALastSemester/MachineLearning/Gapminder_CSV_datasets/data")
]
DATA_DIR = next((p.resolve() for p in DATA_CANDIDATES if (p / "gapminder_full_1952_2007.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục dữ liệu. Hãy đặt bốn tệp CSV trong ../data hoặc data.")
print("Thư mục dữ liệu:", DATA_DIR)

full = pd.read_csv(DATA_DIR / "gapminder_full_1952_2007.csv")
train = pd.read_csv(DATA_DIR / "gapminder_train_1952_1997.csv")
validation = pd.read_csv(DATA_DIR / "gapminder_validation_2002.csv")
test = pd.read_csv(DATA_DIR / "gapminder_test_2007.csv")

def engineer(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["log2_gdp"] = np.log2(out["gdpPercap"])
    out["log2_gdp_sq"] = out["log2_gdp"] ** 2
    out["log10_pop"] = np.log10(out["pop"])
    out["time5"] = (out["year"] - 1952) / 5
    return out

train_e = engineer(train)
validation_e = engineer(validation)
test_e = engineer(test)
full_e = engineer(full)

def regression_metrics(y_true, y_pred) -> pd.Series:
    return pd.Series({
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    })

Thư mục dữ liệu: /content/drive/MyDrive/ALastSemester/MachineLearning/Gapminder_CSV_datasets/data


## Mô-đun 1 — Kiểm toán dữ liệu và phân tích khám phá (15 điểm)

### B1. Hồ sơ bốn tệp dữ liệu — 3 điểm
Tạo bảng gồm: tên tập, số dòng, số cột, số quốc gia, các năm, số giá trị thiếu và số dòng trùng lặp.

In [ ]:
# TODO B1
# Gợi ý: lặp qua một dictionary chứa full/train/validation/test.

### B2. Kiểm chứng phép chia theo thời gian — 2 điểm

1. Chứng minh các cặp `(country, year)` giữa train, validation và test không giao nhau.
2. Kiểm tra hợp của ba tập có đúng bằng tập full hay không.
3. Giải thích vì sao phép chia này phù hợp hơn phép chia ngẫu nhiên cho bài toán dự báo tương lai.

In [ ]:
# TODO B2

### B3. Kiểm tra dữ liệu bảng cân bằng — 2 điểm

Đếm số mốc thời gian của từng quốc gia trong `full` và `train`. Có quốc gia nào thiếu mốc năm hay không?

In [ ]:
# TODO B3

### B4. Từ điển biến và thang đo — 2 điểm

Phân loại từng biến theo: định danh, định tính danh nghĩa, định lượng rời rạc, định lượng liên tục. Chọn biến đích và giải thích vì sao `country` không nên được coi như một số thông thường.

### B5. Phân phối và phép biến đổi — 3 điểm

1. Tính mean, median, standard deviation, min, max và skewness cho `lifeExp`, `gdpPercap`, `pop` trên train.
2. So sánh skewness trước và sau khi dùng log.
3. Vẽ riêng từng biểu đồ: histogram GDP thô, histogram log-GDP, scatter tuổi thọ–GDP thô, scatter tuổi thọ–log-GDP.

In [ ]:
# TODO B5

### B6. Khác biệt theo thời gian và châu lục — 3 điểm

Tạo bảng trung bình tuổi thọ theo năm và theo châu lục; vẽ đường xu hướng trung bình theo năm. Thảo luận liệu một hệ số thời gian chung cho mọi châu lục có hợp lý hay không.

In [ ]:
# TODO B6

## Mô-đun 2 — Hồi quy tuyến tính đơn (20 điểm)

### B7. Mô hình nền trung bình — 3 điểm

Dùng tuổi thọ trung bình của train làm dự báo cho mọi quan sát. Tính MAE, RMSE, R² trên train và validation. Giải thích vì sao R² validation có thể âm.

In [ ]:
# TODO B7

### B8. Tự tính OLS — 4 điểm

Với mô hình `lifeExp = β0 + β1*gdpPercap + ε`, tự tính `β0`, `β1` bằng công thức OLS, sau đó đối chiếu với `LinearRegression` hoặc `statsmodels`.

In [ ]:
# TODO B8

### B9. GDP thô và ý nghĩa hệ số — 3 điểm

Ước lượng mô hình trên train. Diễn giải tác động liên hệ của GDP/người tăng 1.000 đơn vị. Đánh giá validation và quan sát đồ thị phần dư.

In [ ]:
# TODO B9

### B10. Mô hình log-GDP — 4 điểm

Ước lượng `lifeExp ~ log2(gdpPercap)`. Diễn giải hệ số dốc theo câu: “khi GDP/người tăng gấp đôi…”. So sánh với B9 bằng RMSE validation và R² validation.

In [ ]:
# TODO B10

### B11. Dự báo trường hợp Việt Nam — 3 điểm

Dùng mô hình B10 dự báo tuổi thọ Việt Nam năm 2002. Tính sai số có dấu và sai số tuyệt đối. Giải thích nguyên nhân mô hình một biến có thể dự báo thấp đáng kể.

In [ ]:
# TODO B11

### B12. Khoảng tin cậy và khoảng dự báo — 3 điểm

Dùng `statsmodels` tính khoảng tin cậy 95% của trung bình và khoảng dự báo 95% cho một quan sát Việt Nam năm 2002. Giải thích vì sao khoảng dự báo rộng hơn.

In [ ]:
# TODO B12

## Mô-đun 3 — Hồi quy đa biến và kỹ thuật đặc trưng (25 điểm)

### B13. Bổ sung xu hướng thời gian — 4 điểm

Dùng `time5 = (year-1952)/5` và ước lượng `lifeExp ~ log2_gdp + time5`. Diễn giải hai hệ số và so sánh validation với mô hình B10.

In [ ]:
# TODO B13

### B14. Biến giả châu lục — 4 điểm

Thêm `C(continent)` với Africa làm nhóm tham chiếu. Viết đầy đủ phương trình cho một quốc gia châu Á và giải thích hệ số châu lục không phải là tác động nhân quả.

In [ ]:
# TODO B14

### B15. Mô hình đa biến chính — 4 điểm

Ước lượng:

\[
lifeExp = \beta_0 + \beta_1\log_2(gdpPercap)+\beta_2 time5+\beta_3\log_{10}(pop)+\gamma^\top continent+\varepsilon.
\]

Báo cáo hệ số, adjusted R² train và ba chỉ số validation.

In [ ]:
# TODO B15

### B16. Hệ số chuẩn hóa — 3 điểm

Chuẩn hóa các biến số và so sánh độ lớn hệ số. Nêu rõ vì sao không chuẩn hóa biến đích vẫn có thể hữu ích khi cần diễn giải theo đơn vị năm tuổi thọ.

In [ ]:
# TODO B16

### B17. Quan hệ phi tuyến nhưng vẫn tuyến tính theo tham số — 4 điểm

Thêm `log2_gdp_sq = log2_gdp**2`. Kiểm tra dấu của hệ số bậc hai và diễn giải hiện tượng lợi ích biên giảm dần. So sánh validation với B15.

In [ ]:
# TODO B17

### B18. Tương tác theo châu lục — 3 điểm

Cho phép hệ số `log2_gdp` và `time5` khác nhau giữa các châu lục. Dùng validation để đánh giá liệu độ phức tạp tăng thêm có đáng giá hay không.

In [ ]:
# TODO B18

### B19. Hiệu ứng cố định quốc gia — 3 điểm

Thêm biến giả `country` hoặc dùng Ridge với one-hot quốc gia. So sánh hai mục tiêu triển khai:

- dự báo tương lai cho **các quốc gia đã biết**;
- dự báo cho **một quốc gia mới chưa từng xuất hiện**.

In [ ]:
# TODO B19

## Mô-đun 4 — Chẩn đoán và suy luận thống kê (20 điểm)

### B20. Đồ thị chẩn đoán — 4 điểm
Với mô hình B15, vẽ riêng: residual–fitted, Q–Q plot, residual theo năm và histogram phần dư. Chỉ ra các vi phạm giả định có thể có.

In [ ]:
# TODO B20

### B21. Phương sai thay đổi — 3 điểm
Thực hiện kiểm định Breusch–Pagan; so sánh standard error OLS với HC3. Kết luận về suy luận thống kê.

In [ ]:
# TODO B21

### B22. Đa cộng tuyến — 3 điểm
Tính VIF cho ma trận thiết kế của B15. Phân biệt VIF cao của intercept với VIF của các biến giải thích.

In [ ]:
# TODO B22

### B23. Điểm ảnh hưởng — 3 điểm
Tính Cook's distance; liệt kê 10 quan sát ảnh hưởng nhất và tìm hiểu vì sao Kuwait, Rwanda hoặc Cambodia có thể xuất hiện.

In [ ]:
# TODO B23

### B24. Sai số có cấu trúc — 3 điểm
Tính MAE và bias trung bình theo châu lục và theo năm. Mô hình có liên tục dự báo cao hoặc thấp cho nhóm nào không?

In [ ]:
# TODO B24

### B25. Sai số chuẩn phân cụm theo quốc gia — 4 điểm
Do mỗi quốc gia xuất hiện nhiều lần, tính cluster-robust standard errors theo `country` và so sánh với OLS/HC3. Hệ số nào thay đổi kết luận đáng kể?

In [ ]:
# TODO B25

## Mô-đun 5 — Đánh giá theo thời gian và dữ liệu bảng (20 điểm)

### B26. Chia ngẫu nhiên so với chia theo thời gian — 4 điểm
So sánh Mô hình B15 trên một random split 80/20 với đánh giá validation 2002. Giải thích cơ chế gây lạc quan của random split.

In [ ]:
# TODO B26

### B27. Bảng lựa chọn mô hình — 4 điểm
Lập bảng cho ít nhất sáu mô hình: baseline, GDP thô, log-GDP, log-GDP + time, mô hình đa biến, mô hình bậc hai. Chọn mô hình bằng validation theo một quy tắc đã nêu trước.

In [ ]:
# TODO B27

### B28. Đánh giá test một lần — 4 điểm
Sau khi khóa mô hình, đánh giá năm 2007. Báo cáo MAE, RMSE, R², biểu đồ actual–predicted và năm quốc gia có sai số tuyệt đối lớn nhất.

In [ ]:
# TODO B28

### B29. Hiệu năng theo châu lục — 4 điểm
Tính MAE/RMSE theo châu lục trên test. Không diễn giải R² cho nhóm chỉ có rất ít quan sát mà không kèm cảnh báo. Đề xuất một cải tiến mô hình.

In [ ]:
# TODO B29

### B30. Tổng quát hóa sang quốc gia mới — 4 điểm
Dùng `GroupKFold` với nhóm là `country`. So sánh kết quả với temporal split và giải thích hai phép đánh giá trả lời hai câu hỏi triển khai khác nhau.

In [ ]:
# TODO B30

## Mô-đun 6 — Dự án tổng hợp

### P1. Nghiên cứu trường hợp Việt Nam
Phân tích quỹ đạo 1952–2007, so sánh actual–predicted, nhận diện giai đoạn mô hình sai nhiều và viết một báo cáo 800–1.200 từ.

### P2. Mô phỏng “GDP tăng gấp đôi”
Dùng mô hình đa biến để mô phỏng thay đổi dự báo tuổi thọ khi GDP/người tăng gấp đôi. Phải có một mục riêng giải thích vì sao đây **không phải** ước lượng tác động chính sách nhân quả.

### P3. Báo cáo nghiên cứu tái lập
Viết báo cáo theo cấu trúc: câu hỏi, dữ liệu, phương pháp, validation, chẩn đoán, test, giới hạn và hướng mở rộng. Nộp notebook có thể chạy từ đầu đến cuối.

## Tiêu chí chấm chung

- Đúng quy trình train–validation–test: 25%.
- Mã chạy được, tái lập và có kiểm tra dữ liệu: 20%.
- Diễn giải hệ số đúng điều kiện “giữ các biến khác không đổi”: 20%.
- Chẩn đoán giả định và thảo luận dữ liệu bảng: 20%.
- Trình bày rõ, bảng/biểu đồ có nhãn và không diễn giải nhân quả quá mức: 15%.